# Regresión Logística
## Integrantes:
- Martínez Marcelo Ingrid Aylen
- Pérez Evaristo Eris
- Ramírez Venegas Alexa Paola

Para esta práctica generaremos y entrenaremos modelos de regresión logística, perceptrones y regresión lineal.

1. Genera los nodos de una gráfica computacional basada en una super clase

In [ ]:
import numpy as np
class Node:
  """ Clase nodo """
  def __call__ (self, x):
    return self.forward(x)

  def __str__(self):
    return str(self.out)

2. El nodo principal para los tres modelos sera el nodo de PreActivation que define la funcion de preactivación $wx + b$

In [ ]:

class PreActivacion(Node):
  """
  Clase del nodo principal de los tres modelos que define la función de preactivación
  """

  def __init__(self,input_dim):
    super().__init__()
    self.w = np.random.randn(input_dim, 1)
    self.b = 1.0
    self.x = None

  def forward(self, x):
    self.x = x
    self.out = np.dot(x, self.w) + self.b
    return self.out

  def backward(self, grad=1):
    #Según recuerdo lo que dijo el ayudante tanto w como b tienen que ser recalculados en el backward pero no sé como hacerlo como los demás nodos.
    #Como aquí con grad de b como de w
    self.grad_w = np.dot(self.x.T, grad)
    self.grad_b = np.sum(grad, axis=0)
    return np.dot(grad, self.w.T)


3. Define el nodo de la funcion Sigmoide para la regresión logística.

In [ ]:
class NodoSigmoide(Node):
  def __init__(self):
    super().__init__()
    self.out = None
    self.input = None

  def forward(self, x):
    self.input = x
    self.out = 1 / (1 + np.exp(-x))
    return self.out

  def backward(self, grad=1):
    if grad.ndim == 1:
      grad = grad.reshape(-1, 1)
    return self.out * (1 - self.out) * grad #Derivada (1 - la sigmoide)

4. Define el nodo para la función de entropía cruzada binaria cuya función es $-yln(f) + (1-y)ln(1-f)$

$y = etiqueta real$
$f = predecida$

In [ ]:
class NodoEntropiaCruzadaB(Node):
  def __init__(self):
    super().__init__()
    self.x = None # Label real
    self.y = None # Predecido

  def __call__(self, x, y):
    return self.forward(x, y)

  def forward(self, x, y):
    """
    x = labelReal
    y = prediccion
    """
    epsilon = 1e-9
    self.x = x
    self.y = np.clip(y, epsilon, 1-epsilon)
    self.out = - np.mean(x * np.log(self.y) + (1 - x)* np.log(1-self.y))
    # Hay problemas si se calcula log(0) con la funcion normal
    #self.out = - (x * np.log(y) + (1 - x) * np.log(1 - y))
    return self.out

  def backward(self, grad=1):
    # Tal vez aquí nos haga falta como hacer algo por separado para x y w?

    # No se si esta bien
    #return (1 - self.y) / (1 - self.x) - self.y / self.x

    return (self.y - self.x) / (self.y * (1 - self.y)) / len(self.x)

5. Entrena el modelo de regresión logística usando datos a partir de sklearn:


Utiliza 100 épocas y una tasa de aprendizaje de 0.1



In [ ]:
class ModeloRegresionLogistica:
  def __init__(self, input_dim):
    self.preact = PreActivacion(input_dim)
    self.sigmoide = NodoSigmoide()

  def forward(self, x):
    z = self.preact(x)
    return self.sigmoide(z)

  def predict(self, x):
      predicciones = self.forward(x)
      return (predicciones >= 0.5).astype(int)

  def train(self, x_train, y_train, epochs = 100, lr = 0.1):
    y_train = y_train.reshape(-1, 1)

    for epoch in range(epochs):
      y_pred = self.forward(x_train)

      # Calcular pérdida
      loss_fun = NodoEntropiaCruzadaB()
      loss = loss_fun(y_pred, y_train)

      # Backward de calculo de gradientes
      gradiente_loss = loss_fun.backward()
      gradiente_sigmoide = self.sigmoide.backward(gradiente_loss)
      grad_pre = self.preact.backward(gradiente_sigmoide)

      # Actualizacipon de pesos
      self.preact.w -= lr * self.preact.grad_w
      self.preact.b -= lr * self.preact.grad_b

      if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

In [ ]:
# Entrenamiento para Regresión Logistica
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch

# Datos
x, y = make_classification(n_samples=1000, n_features=2, n_redundant=0, n_informative=2, random_state=10)

# División
x_train, x_eval, y_train, y_eval = train_test_split(x,y,test_size=0.3)

regLog = ModeloRegresionLogistica(input_dim=2)
regLog .train(x_train, y_train, epochs=100, lr=0.1)

# Evaluar con el reporte de clasificación sklearn
yPrediccionRegLog = regLog .predict(x_eval)
print("\nReporte de clasificación - Modelo1: Regresión Logística:")
print(classification_report(y_eval, yPrediccionRegLog))
print("Pesos finales:", regLog .preact.w)
print("Sesgo final:", regLog .preact.b)

# Falta Verificaaar

Epoch 0, Loss: 10.3211
Epoch 10, Loss: 17.8516
Epoch 20, Loss: 17.8516
Epoch 30, Loss: 17.8516
Epoch 40, Loss: 17.8516
Epoch 50, Loss: 17.8516
Epoch 60, Loss: 17.8516
Epoch 70, Loss: 17.8516
Epoch 80, Loss: 17.8516
Epoch 90, Loss: 17.8516

Reporte de clasificación - Modelo1: Regresión Logística:
              precision    recall  f1-score   support

           0       0.08      0.07      0.07       144
           1       0.22      0.24      0.23       156

    accuracy                           0.16       300
   macro avg       0.15      0.16      0.15       300
weighted avg       0.15      0.16      0.16       300

Pesos finales: [[-6862132.59134999]
 [ 2011045.56242479]]
Sesgo final: [2818483.70809822]


/tmp/ipython-input-2468748848.py:9: RuntimeWarning: overflow encountered in exp
  self.out = 1 / (1 + np.exp(-x))


6. Evalua el resultado usando el reporte de clasificación

In [ ]:
from sklearn.metrics import classification_report

7. Define un nodo computacional para la función escalonada del perceptrón y comprueba que la clasificación es la misma que con regresión logística evaluando el resultado.

8. Define un nodo para la función objetivo del error cuadrático y entrena un modelo de regresión lineal $f(x) = wx + b$ con los datos:



```
  from sklearn.datasets import make_regression

  x. y = make_regression(n_samples=1000, n_features=2, n_informative=2)
  x_train, x_eval, y_train, y_eval = train_test_split(x, y, test_size=0.3)
```

Evalúa usando mean_squared_error y r2_score de sklearn.

9. Recuerda que cada nodo debe de tener una función forward que computa la función y otra backward que computa el gradiente. Los pesos que se actualizarán serán los de pre-activación:


```
  pre.w -= lr.a.grad
  pre.b -= lr.a.grad_b
```